# 🤖 AI Model Demo Inference — Flood Rescue System

> **Project**: Edge AI–Based System for Multimodal Analysis and Clustering of Flood Rescue Events
>
> **Purpose**: Demo inference with pre-trained models (NO training) for Progress Report #1
>
> **Environment**: Google Colab (T4 GPU) / Kaggle (P100 GPU)

## 1. Environment Setup

In [ ]:
!pip install -q torch torchvision timm transformers sentencepiece Pillow pandas matplotlib seaborn tqdm

In [ ]:
import os, time, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_DIR = Path('dataset')
REPORT_DIR = BASE_DIR / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   PyTorch: {torch.__version__}')

## 2. Load Sample Images

In [ ]:
# Load 10 balanced sample images from metadata.csv when available
CLASSES = ['no_flood', 'low_flood', 'high_flood']
metadata_path = BASE_DIR / 'metadata.csv'

if metadata_path.exists():
    metadata_df = pd.read_csv(metadata_path)
    sample_df = (
        metadata_df.groupby('mapped_label', group_keys=False)
        .apply(lambda frame: frame.sample(min(4, len(frame)), random_state=SEED))
        .reset_index(drop=True)
    )
    sample_df = sample_df.sort_values(['mapped_label', 'source']).head(10).reset_index(drop=True)
    sample_images = sample_df['processed_path'].tolist()
    sample_labels = sample_df['mapped_label'].tolist()
    sample_sources = sample_df['source'].tolist()
else:
    SAMPLE_DIR = Path('dataset/image_data/test')
    sample_images, sample_labels, sample_sources = [], [], []
    for cls in CLASSES:
        cls_dir = SAMPLE_DIR / cls
        imgs = sorted(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))[:4] if cls_dir.exists() else []
        if len(imgs) == 0:
            os.makedirs(f'sample_images/{cls}', exist_ok=True)
            for i in range(4):
                colors = {'no_flood': (34, 139, 34), 'low_flood': (70, 130, 180), 'high_flood': (139, 69, 19)}
                c = colors[cls]
                img = Image.new('RGB', (224, 224), color=(c[0] + random.randint(-20, 20), c[1] + random.randint(-20, 20), c[2] + random.randint(-20, 20)))
                p = Path(f'sample_images/{cls}/{cls}_{i}.jpg')
                img.save(p)
                imgs.append(p)
        for p in imgs[:4]:
            sample_images.append(str(p))
            sample_labels.append(cls)
            sample_sources.append('fallback_sample')

sample_manifest = pd.DataFrame({
    'image_path': sample_images,
    'true_label': sample_labels,
    'source': sample_sources,
})
sample_manifest.to_csv(REPORT_DIR / 'image_inference_manifest.csv', index=False, encoding='utf-8-sig')

print(f'📸 Loaded {len(sample_images)} sample images')
print(sample_manifest['true_label'].value_counts())

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Sample Images for Inference', fontsize=16, fontweight='bold')
for i in range(min(10, len(sample_images))):
    ax = axes[i // 5][i % 5]
    img = Image.open(sample_images[i]).convert('RGB').resize((224, 224))
    ax.imshow(img)
    ax.set_title(f"{sample_labels[i]}\n{sample_sources[i]}", fontsize=9)
    ax.axis('off')
for j in range(len(sample_images), 10):
    axes[j // 5][j % 5].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Load ImageNet labels
import urllib.request
url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
try:
    urllib.request.urlretrieve(url, 'imagenet_classes.txt')
    with open('imagenet_classes.txt') as f:
        imagenet_labels = [line.strip() for line in f.readlines()]
    print(f"✅ Loaded {len(imagenet_labels)} ImageNet labels")
except:
    imagenet_labels = [f'class_{i}' for i in range(1000)]
    print("⚠️ Using placeholder labels")

In [ ]:
# Common preprocessing and inference functions
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def get_model_size(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def run_inference(model, image_paths, labels, sources, model_name, n_runs=5):
    model.eval()
    results = []

    model_gpu = model.to(device)
    gpu_times = []
    for img_path, label, source in zip(image_paths, labels, sources):
        img = Image.open(img_path).convert('RGB')
        input_tensor = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            _ = model_gpu(input_tensor)

        times = []
        for _ in range(n_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                output = model_gpu(input_tensor)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)

        probs = torch.nn.functional.softmax(output[0], dim=0)
        top5_prob, top5_idx = torch.topk(probs, 5)
        top5 = [(imagenet_labels[idx.item()], prob.item()) for idx, prob in zip(top5_idx, top5_prob)]
        gpu_time = float(np.median(times))
        gpu_times.append(gpu_time)

        results.append({
            'image': Path(img_path).name,
            'source': source,
            'true_label': label,
            'top1': f"{top5[0][0]} ({top5[0][1]:.3f})",
            'top2': f"{top5[1][0]} ({top5[1][1]:.3f})",
            'top3': f"{top5[2][0]} ({top5[2][1]:.3f})",
            'gpu_ms': round(gpu_time, 2),
        })

    model_cpu = model.cpu()
    cpu_times = []
    for img_path in image_paths[:3]:
        img = Image.open(img_path).convert('RGB')
        input_tensor = preprocess(img).unsqueeze(0)
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            with torch.no_grad():
                _ = model_cpu(input_tensor)
            times.append((time.perf_counter() - t0) * 1000)
        cpu_times.append(float(np.median(times)))

    avg_gpu = float(np.mean(gpu_times)) if gpu_times else float('nan')
    avg_cpu = float(np.mean(cpu_times)) if cpu_times else float('nan')
    n_params = count_params(model)
    size_mb = get_model_size(model)
    results_df = pd.DataFrame(results)
    results_df.to_csv(REPORT_DIR / f"{model_name.lower().replace('-', '_').replace(' ', '_')}_predictions.csv", index=False, encoding='utf-8-sig')

    print(f"\n{'=' * 70}")
    print(f"📊 {model_name}")
    print(f"{'=' * 70}")
    print(f"  Parameters:  {n_params:,}")
    print(f"  Size:        {size_mb:.1f} MB")
    print(f"  Avg GPU:     {avg_gpu:.1f} ms")
    print(f"  Avg CPU:     {avg_cpu:.1f} ms")
    display(results_df.head(10))

    return {
        'model': model_name,
        'params': n_params,
        'size_mb': size_mb,
        'avg_gpu_ms': avg_gpu,
        'avg_cpu_ms': avg_cpu,
        'results': results,
        'results_df': results_df,
    }

## 3. MobileNetV3-Small (ImageNet)

In [ ]:
print('📥 Loading MobileNetV3-Small...')
mobilenet_v3_small = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
metrics_mv3s = run_inference(mobilenet_v3_small, sample_images, sample_labels, sample_sources, 'MobileNetV3-Small')

## 4. MobileNetV3-Large (ImageNet)

In [ ]:
print('📥 Loading MobileNetV3-Large...')
mobilenet_v3_large = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
metrics_mv3l = run_inference(mobilenet_v3_large, sample_images, sample_labels, sample_sources, 'MobileNetV3-Large')

## 5. EfficientNet-Lite0 (via timm)

In [ ]:
import timm
print('📥 Loading EfficientNet-Lite0...')
try:
    efficientnet = timm.create_model('efficientnet_lite0', pretrained=True)
    efficientnet_name = 'EfficientNet-Lite0'
except Exception:
    print('⚠️ efficientnet_lite0 not found, using efficientnet_b0 as fallback')
    efficientnet = timm.create_model('efficientnet_b0', pretrained=True)
    efficientnet_name = 'EfficientNet-B0-fallback'
metrics_eff = run_inference(efficientnet, sample_images, sample_labels, sample_sources, efficientnet_name)

## 6. ResNet-18 (Baseline)

In [ ]:
print('📥 Loading ResNet-18...')
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
metrics_r18 = run_inference(resnet18, sample_images, sample_labels, sample_sources, 'ResNet-18')

## 7. Image Model Comparison

In [ ]:
# Comparison table
all_metrics = [metrics_mv3s, metrics_mv3l, metrics_eff, metrics_r18]
comparison = pd.DataFrame([{
    'Model': m['model'],
    'Parameters': f"{m['params']:,}",
    'Size (MB)': f"{m['size_mb']:.1f}",
    'Avg GPU (ms)': f"{m['avg_gpu_ms']:.1f}",
    'Avg CPU (ms)': f"{m['avg_cpu_ms']:.1f}",
} for m in all_metrics])

print("\n" + "="*80)
print("📊 IMAGE MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
names = [m['model'] for m in all_metrics]
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

ax = axes[0]
sizes = [m['size_mb'] for m in all_metrics]
ax.bar(names, sizes, color=colors, edgecolor='black')
ax.set_title('Model Size (MB)', fontweight='bold')
for i, v in enumerate(sizes): ax.text(i, v+0.5, f'{v:.1f}', ha='center')
ax.tick_params(axis='x', rotation=15)

ax = axes[1]
gpu = [m['avg_gpu_ms'] for m in all_metrics]
ax.bar(names, gpu, color=colors, edgecolor='black')
ax.set_title('GPU Inference (ms)', fontweight='bold')
for i, v in enumerate(gpu): ax.text(i, v+0.2, f'{v:.1f}', ha='center')
ax.tick_params(axis='x', rotation=15)

ax = axes[2]
cpu = [m['avg_cpu_ms'] for m in all_metrics]
ax.bar(names, cpu, color=colors, edgecolor='black')
ax.set_title('CPU Inference (ms)', fontweight='bold')
for i, v in enumerate(cpu): ax.text(i, v+1, f'{v:.1f}', ha='center')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('model_comparison_image.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Comparison chart saved")

## 8. Text Models — PhoBERT

Load `vinai/phobert-base` and verify Vietnamese tokenization on rescue messages.

In [ ]:
from transformers import AutoModel, AutoTokenizer

text_path = BASE_DIR / 'text_data' / 'rescue_text_samples.csv'
if text_path.exists():
    text_df = pd.read_csv(text_path)
    text_demo_df = (
        text_df.groupby('urgency_label', group_keys=False)
        .apply(lambda frame: frame.sample(min(3, len(frame)), random_state=SEED))
        .reset_index(drop=True)
    )
else:
    text_demo_df = pd.DataFrame({
        'raw_text': [
            'Cứu với! Nước dâng nóc nhà rồi, có bà già 80 tuổi',
            'Nhà em cần gạo và nước uống, nước ngập nhưng chưa nguy hiểm',
            'Gia đình em đã sơ tán an toàn, cảm ơn mọi người',
            'Dự báo thời tiết ngày mai trời nắng đẹp',
        ],
        'urgency_label': ['urgent_rescue', 'need_supplies', 'safe_update', 'irrelevant'],
    })

print('📥 Loading PhoBERT-base...')
t0 = time.time()
phobert_tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')
phobert_model = AutoModel.from_pretrained('vinai/phobert-base')
phobert_load_time = time.time() - t0
phobert_param_count = sum(p.numel() for p in phobert_model.parameters())
phobert_size_mb = sum(p.numel() * p.element_size() for p in phobert_model.parameters()) / (1024 ** 2)
print(f'✅ PhoBERT loaded in {phobert_load_time:.1f}s')
print(f'   Vocab size: {phobert_tokenizer.vocab_size:,}')
print(f'   Parameters: {phobert_param_count:,}')
print(f'   Size: {phobert_size_mb:.1f} MB')

phobert_model.eval()
phobert_model.to(device)
phobert_rows = []
for _, row in text_demo_df.head(10).iterrows():
    text = row['raw_text']
    inputs = phobert_tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = phobert_model(**inputs)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000
    phobert_rows.append({
        'input_text': text,
        'expected_label': row['urgency_label'],
        'token_count': int(inputs['input_ids'].shape[1]),
        'embedding_shape': tuple(outputs.last_hidden_state.shape),
        'inference_ms': round(t_ms, 2),
    })

phobert_results_df = pd.DataFrame(phobert_rows)
phobert_results_df.to_csv(REPORT_DIR / 'phobert_demo_results.csv', index=False, encoding='utf-8-sig')
display(phobert_results_df)
phobert_avg_ms = phobert_results_df['inference_ms'].mean() if len(phobert_results_df) else float('nan')
print(f'\n📊 PhoBERT avg inference: {phobert_avg_ms:.1f} ms')

## 9. XLM-RoBERTa — Zero-Shot Classification

Using `joeddav/xlm-roberta-large-xnli` for zero-shot rescue message classification (NO training needed).

In [ ]:
from transformers import pipeline

print('📥 Loading XLM-RoBERTa for zero-shot classification...')
t0 = time.time()
classifier = pipeline(
    'zero-shot-classification',
    model='joeddav/xlm-roberta-large-xnli',
    device=0 if torch.cuda.is_available() else -1,
 )
xlmr_load_time = time.time() - t0
print(f'✅ Loaded in {xlmr_load_time:.1f}s')

candidate_labels = ['urgent rescue', 'need supplies', 'safe update', 'irrelevant']
zero_shot_rows = []
correct = 0
for _, row in text_demo_df.head(10).iterrows():
    text = row['raw_text']
    expected = row['urgency_label']
    t0 = time.perf_counter()
    result = classifier(text, candidate_labels)
    t_ms = (time.perf_counter() - t0) * 1000
    predicted = result['labels'][0]
    predicted_mapped = predicted.replace(' ', '_')
    if predicted_mapped == expected:
        correct += 1
    zero_shot_rows.append({
        'input_text': text,
        'expected_label': expected,
        'predicted_label': predicted_mapped,
        'confidence': round(float(result['scores'][0]), 4),
        'second_choice': result['labels'][1].replace(' ', '_'),
        'inference_ms': round(t_ms, 2),
    })

zero_shot_df = pd.DataFrame(zero_shot_rows)
zero_shot_df.to_csv(REPORT_DIR / 'xlmr_zero_shot_results.csv', index=False, encoding='utf-8-sig')
display(zero_shot_df)
print(f"\n📊 Zero-shot accuracy on demo set: {correct}/{len(zero_shot_df)} ({(correct / max(len(zero_shot_df), 1)) * 100:.0f}%)")
print(f"📊 Avg inference: {zero_shot_df['inference_ms'].mean():.1f} ms")

## 10. Vietnamese Sentiment Analysis

In [ ]:
# Try loading a Vietnamese sentiment model
print("📥 Trying Vietnamese sentiment models...")
vn_sentiment = None

model_names = [
    "wonrax/phobert-base-vietnamese-sentiment",
    "cardiffnlp/twitter-xlm-roberta-base-sentiment",
]

for mname in model_names:
    try:
        print(f"  Trying: {mname}")
        vn_sentiment = pipeline("sentiment-analysis", model=mname,
                               device=0 if torch.cuda.is_available() else -1)
        print(f"  ✅ Loaded: {mname}")
        break
    except Exception as e:
        print(f"  ⚠️ Failed: {e}")

if vn_sentiment:
    test_vn = [
        "Cứu với nước ngập hết rồi",
        "Cảm ơn mọi người đã giúp đỡ",
        "Tức quá kêu cứu mà không ai đến",
        "Gia đình em đã an toàn",
        "Hết thức ăn rồi, cần tiếp tế gấp",
    ]
    print("\n📊 Vietnamese Sentiment Results:")
    for text in test_vn:
        result = vn_sentiment(text)[0]
        print(f"  {text:<45s} → {result['label']} ({result['score']:.3f})")
else:
    print("\n⚠️ No Vietnamese sentiment model available")
    print("   PhoBERT embeddings above can be used for future fine-tuning")

## 11. Text Model Comparison

In [ ]:
# Text model comparison table
text_models = pd.DataFrame([
    {
        'Model': 'PhoBERT-base',
        'Load Time (s)': f'{phobert_load_time:.1f}',
        'Vocab Size': f'{phobert_tokenizer.vocab_size:,}',
        'Parameters': f'{phobert_param_count:,}',
        'Size (MB)': f'{phobert_size_mb:.1f}',
        'Avg Inference (ms)': f"{phobert_results_df['inference_ms'].mean():.1f}",
        'Vietnamese': '✅ Native',
    },
    {
        'Model': 'XLM-RoBERTa-XNLI',
        'Load Time (s)': f'{xlmr_load_time:.1f}',
        'Vocab Size': 'N/A',
        'Parameters': f"{classifier.model.num_parameters():,}" if hasattr(classifier.model, 'num_parameters') else 'N/A',
        'Size (MB)': 'N/A',
        'Avg Inference (ms)': f"{zero_shot_df['inference_ms'].mean():.1f}",
        'Vietnamese': '✅ Multilingual',
    },
])
text_models.to_csv(REPORT_DIR / 'text_model_comparison.csv', index=False, encoding='utf-8-sig')
print('\n' + '=' * 80)
print('📊 TEXT MODEL COMPARISON')
print('=' * 80)
display(text_models)

print('\n📝 Qualitative Observations:')
print('  1. PhoBERT tokenizes Vietnamese text naturally and preserves diacritics.')
print('  2. XLM-RoBERTa zero-shot provides a no-training baseline for urgency classification.')
print('  3. PhoBERT should be the fine-tuning candidate, while zero-shot remains useful for rapid validation.')

## 12. TFLite Conversion Demo (Bonus)

In [ ]:
# Export MobileNetV3-Small → ONNX → TFLite size estimate
print("📦 Model Conversion Demo")
print("="*60)

# Save PyTorch model
mobilenet_v3_small.eval().cpu()
dummy = torch.randn(1, 3, 224, 224)

# Export to ONNX
try:
    onnx_path = 'mobilenetv3_small.onnx'
    torch.onnx.export(mobilenet_v3_small, dummy, onnx_path,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input':{0:'batch'}, 'output':{0:'batch'}})
    onnx_size = os.path.getsize(onnx_path) / (1024**2)
    print(f"  ✅ ONNX export: {onnx_size:.1f} MB")
except Exception as e:
    print(f"  ⚠️ ONNX export failed: {e}")
    onnx_size = 0

# TFLite conversion (if TF available)
try:
    import tensorflow as tf
    print("  Converting ONNX → TFLite...")

    # Use onnx-tf or estimate
    # For demo, we estimate TFLite sizes
    pytorch_size = get_model_size(mobilenet_v3_small)
    fp16_est = pytorch_size * 0.5
    int8_est = pytorch_size * 0.25

    print(f"\n  📊 Model Size Comparison:")
    print(f"  {'Format':<20s} {'Size (MB)'}")
    print(f"  {'-'*30}")
    print(f"  {'PyTorch (FP32)':<20s} {pytorch_size:.1f}")
    print(f"  {'ONNX':<20s} {onnx_size:.1f}")
    print(f"  {'TFLite FP16 (est.)':<20s} {fp16_est:.1f}")
    print(f"  {'TFLite INT8 (est.)':<20s} {int8_est:.1f}")
except ImportError:
    pytorch_size = get_model_size(mobilenet_v3_small)
    print(f"\n  ⚠️ TensorFlow not installed. Size estimates:")
    print(f"  {'Format':<20s} {'Size (MB)'}")
    print(f"  {'-'*30}")
    print(f"  {'PyTorch (FP32)':<20s} {pytorch_size:.1f}")
    print(f"  {'ONNX':<20s} {onnx_size:.1f}")
    print(f"  {'TFLite FP16 (est.)':<20s} {pytorch_size*0.5:.1f}")
    print(f"  {'TFLite INT8 (est.)':<20s} {pytorch_size*0.25:.1f}")

print("\n  💡 For mobile deployment:")
print("  • MobileNetV3-Small INT8 ≈ 1-2 MB — ideal for on-device inference")
print("  • Full conversion requires: pip install tensorflow onnx-tf")

## 13. Summary & Next Steps

In [ ]:
comparison.to_csv(REPORT_DIR / 'image_model_comparison.csv', index=False, encoding='utf-8-sig')

print('=' * 80)
print('📊 FINAL SUMMARY — AI Model Demo Inference')
print('=' * 80)

print('\n🖼️  IMAGE MODELS:')
display(comparison)

print('\n📝 TEXT MODELS:')
display(text_models)

print('\n🎯 KEY FINDINGS:')
print('  1. MobileNetV3-Small remains the strongest edge candidate thanks to its size/latency profile.')
print('  2. ImageNet pre-training captures partial flood semantics but still needs flood-specific fine-tuning.')
print('  3. PhoBERT is the most suitable Vietnamese backbone for downstream rescue text fine-tuning.')
print('  4. Zero-shot XLM-RoBERTa is a useful qualitative baseline for Progress Report #1.')

print('\n📁 Generated report artifacts:')
for name in [
    'image_inference_manifest.csv',
    'mobilenetv3_small_predictions.csv',
    'mobilenetv3_large_predictions.csv',
    'resnet_18_predictions.csv',
    'image_model_comparison.csv',
    'phobert_demo_results.csv',
    'xlmr_zero_shot_results.csv',
    'text_model_comparison.csv',
]:
    print(f'  - {REPORT_DIR / name}')

print('\n✅ Demo inference notebook is ready to run end-to-end on Colab or Kaggle.')